In [1]:

import numpy as np
import pandas as pd
import os
import rioxarray as rxr
from osgeo import gdal
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline, make_pipeline
import geopandas as gpd
import rasterio
from shapely.geometry import box
from shapely.geometry import Point
from rasterio.sample import sample_gen
import xgboost as xgb

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from skopt import BayesSearchCV
from skopt.space import Integer, Real

c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\cupy\_environment.py:215: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


In [4]:
def save_img(output_filename, reference_filename, array):
    
    reference = gdal.Open(reference_filename)
    
    # 获取地理变换参数和坐标系
    geo_transform = reference.GetGeoTransform()
    projection = reference.GetProjection()

    # 获取矩阵尺寸
    rows, cols = array.shape

    # 创建新的 TIFF 数据集
    driver = gdal.GetDriverByName('GTiff')
    out_data = driver.Create(output_filename, cols, rows, 1, gdal.GDT_Float32)

    # 设置地理变换和坐标系
    out_data.SetGeoTransform(geo_transform)
    out_data.SetProjection(projection)
    

    # 写入数据
    out_data.GetRasterBand(1).WriteArray(array)

    # 保存并关闭文件
    out_data.FlushCache()
    out_data = None
    
def load_refer_icesat_point(FB_raster, icesat_csv_path, output_point, output_raw_point):
    
    df = pd.read_csv(icesat_csv_path)
    lon, lat = 'mean_lon', 'mean_lat'
    # LULC_path = "E:/Downloads/tes/UK_LULC.tif"
    geometry = [Point(xy) for xy in zip(df[lon], df[lat])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry)
    gdf.set_crs(epsg=4326, inplace=True)
    points = gdf
    
    with rasterio.open(FB_raster) as src:
        
        if points.crs != src.crs:
            points = points.to_crs(src.crs)
        coords = [(point.x, point.y) for point in points.geometry]
        values = list(src.sample(coords))
        
        band_count = src.count
        for band in range(band_count):
            points[f"b{band+1}_FB6"] = [v[band] for v in values]
            
    # with rasterio.open(LULC_path) as src:

    #     if points.crs != src.crs:
    #         points = points.to_crs(src.crs)
    #     coords = [(point.x, point.y) for point in points.geometry]
    #     values = list(src.sample(coords))

    #     band_count = src.count
    #     points['LULC'] = [v[0] for v in values] 
     
    icesat_mask = (points['OH'] < 5) & (points['b7_FB6'] > 0) #& ((points['LULC'] == 200) | (points['LULC'] == 80))
    data = points[icesat_mask]
    
    # points.to_file(output_raw_point)
    # data.to_file(output_point)
    
    return data

def optimize_random_forest(x_train, y_train,  n_iter=30, cv=3):

    # if feature_columns is None:
    #     raise ValueError("feature_columns must be provided.")

    classifier = RandomForestRegressor(random_state=42)

    search_space = {
        'n_estimators': Integer(50, 500),
        'max_depth': Integer(3, 20),
        'min_samples_split': Integer(2, 20),
        'min_samples_leaf': Integer(1, 20),
        'max_features': Real(0.1, 1.0)
    }

    opt = BayesSearchCV(
        estimator=classifier,
        search_spaces=search_space,
        n_iter=n_iter,
        cv=cv,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1
    )

    opt.fit(x_train, y_train)

    best_params = opt.best_params_
    print('Best hyperparameters:', best_params)
    print('Best score (negative MSE):', opt.best_score_)

    # 使用最优参数重新训练模型
    final_model = RandomForestRegressor(**best_params, random_state=42)
    # final_model.fit(x_train, y_train)

    return best_params, final_model


In [8]:
#----------------------------all ICESat-2 used to train RF modal -----------------------------------------
# Base info

region = 'Region1'

#location_list = ['Region5a', 'Region5b','Region5c', 'Region5d','Region5e']
# location_list = ['Region6a', 'Region6d','Region6e', 'Region6f','Region6g','Region6h','Region6i','Region6j']
# location_list = ['Thames', 'Region1a','Region1b', 'Region1c']
location_list = ['Thames1']
location2 = 'Thames'
year_list = [2020]

method = 'RF'
band = 4



for i in range(len(location_list)):
    
    location = location_list[i]
    
    target_band, target_id = ['b1_FB6','b2_FB6','b3_FB6','b4_FB6'], [0, 1, 2, 3]
    #target_band, target_id = ['b1_FB6'], [0]
    
    # load classifer  
    ICESat_points_list = []
    for k in range(len(year_list)):
        
        
        FB_raster_path_ = f"G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}/{location}_{year_list[k]}_FB6_new9.tif"
        
        ICESAT_csv_path_ = f'G:\ICESat-2\{region}\{region}_{location2}\A_{location2}_30m_{year_list[k]}_new10.csv'
        output_raw_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_raw_new10.shp'
        output_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_new10.shp'
        data = load_refer_icesat_point(FB_raster_path_, ICESAT_csv_path_, output_point, output_raw_point)
            
        ICESat_points_list.append(data.to_crs(epsg=4326))
            
    data = pd.concat(ICESat_points_list)
    y_train = data['OH'].values
    x_train = data[target_band].values
    
    if method == 'RF':
        best_params, classifier = optimize_random_forest(x_train, y_train)
        classifier.fit(x_train, y_train)
    elif method == 'Linear':
        classifier = np.polyfit(x_train[:, 0], y_train, deg=1)
    else:
        classifier = np.polyfit(x_train[:, 0], y_train, deg=2)
    
    for j in range(len(year_list)):
      
        # Input info 
        FB_raster_path = f"G:\ICESat-2\{region}_map\{location}_map\A{year_list[j]}/{location}_{year_list[j]}_FB6_new9.tif"
    
        # Output Info
        output_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[j]}\{location}_{year_list[j]}_{band}_{method}_FB6_new9_bys_SY.tif'


        if not os.path.exists(FB_raster_path):
            print(FB_raster_path, ' is not exist!')
            break
       

        # Elevation mapping
        raster = rxr.open_rasterio(FB_raster_path).values
        raster = np.nan_to_num(raster, nan = 0)
        selected_raster = raster[target_id, :, :]

        if len(selected_raster.shape) != 3:
            row, col = selected_raster.shape
            selected_raster = selected_raster.reshape(1, row, col)
        else: 
            bands, row, col = selected_raster.shape

        print('selected_raster.shape', selected_raster.shape)

        k = 0
        result = np.array([[]])
        for i in range(row):
            
            x_train = np.transpose(selected_raster[:, i, :])
            if method == 'RF':
                y_pred = classifier.predict(x_train)
            elif method == 'Linear':
                y_pred = np.polyval(classifier, x_train)
            else:
                y_pred = np.polyval(classifier, x_train)
            
            result = np.append(result, y_pred)
            
            if i%(int(row/5)) == 0:
                
                print(k*20 , '%_ is done')
                k = k+1
        
        result = result.reshape(row, col)
        print(result.shape, type(result))
        # tf_mask = raster[6,:,:].flatten()
        # result[tf_mask == 0] = 0
        # result_ = result.reshape(row, col)
        tf_mask = raster[6,:,:]#.flatten()
        # Freq_s2 = raster[5,:,:]#.flatten()
        
        z = rxr.open_rasterio(FB_raster_path, masked=True).squeeze() 
        
        #result[(tf_mask == 0) | ((LULC_aligned != 200) & (LULC_aligned != 80))] = 0
        result[(tf_mask == 0)] = 0       

        save_img(output_path, FB_raster_path, result)
        print(f'{location}_{year_list[j]} is output to {output_path}')

Best hyperparameters: OrderedDict([('max_depth', 13), ('max_features', 0.10630866491032005), ('min_samples_leaf', 1), ('min_samples_split', 7), ('n_estimators', 69)])
Best score (negative MSE): -0.11755361407727588
selected_raster.shape (4, 1728, 1575)
0 %_ is done
20 %_ is done
40 %_ is done
60 %_ is done
80 %_ is done
100 %_ is done
(1728, 1575) <class 'numpy.ndarray'>
Thames1_2020 is output to G:\ICESat-2\Region1_map\Thames1_map\A2020\Thames1_2020_4_RF_FB6_new9_bys_SY.tif


In [ ]:
#----------------------------all ICESat-2 used to train RF modal -----------------------------------------
# Base info
region = 'Region1'

#location_list = ['Region5a', 'Region5b','Region5c', 'Region5d','Region5e']
#location_list = ['Region6a', 'Region6b','Region6c', 'Region6d','Region6e']
location_list = ['Thames', 'Region1a','Region1b', 'Region1c']
location_list = ['Thames']
# location_list = ['Solway']
year_list = [2019, 2020, 2021, 2022, 2023, 2024]
#year_list = [2020, 2021 ,2022, 2023, 2024]
# region = 'Region2'
# location_list = ['wash_bay', 'Henber', 'Region2a']
# year_list = [2019, 2020, 2021, 2022, 2023, 2024]
# location_list = ['Thames']
# year_list = [2019]
method = 'RF'
band = 4

for i in range(len(location_list)):
    
    location = location_list[i]
    
    LULC_path = "E:/Downloads/tes/UK_LULC.tif"
    LULC = rxr.open_rasterio(LULC_path, masked=True).squeeze()
    target_band_list, target_id_list = ['b1_FB6','b2_FB6','b3_FB6','b4_FB6'], [0, 1, 2, 3]
    target_band, target_id = target_band_list[:band], target_id_list[:band]
    # target_band, target_id = ['b1_FB6'], [0]
    
    
    # load classifer  
    ICESat_points_list = []
    for k in range(len(year_list)):
        
        
        FB_raster_path_ = f"G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}/{location}_{year_list[k]}_FB6_new7.tif"
        
        ICESAT_csv_path_ = f'G:\ICESat-2\{region}\{region}_{location}\A_{location}_30m_{year_list[k]}_new10.csv'
        output_raw_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_raw_new10.shp'
        output_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_new10.shp'
        
        data = load_refer_icesat_point(FB_raster_path_, ICESAT_csv_path_, output_point, output_raw_point)
            
        ICESat_points_list.append(data)
        
    #----------------------------------------------------------------------------------------------
    
    # for k in range(len(year_list)):
        
    #     point_path = f'G:\ICESat-2\Discussion5_1\ICESAT\Ribble\A_{location}_c_30m_{year_list[k]}_new10.shp'
    #     point = gpd.read_file(point_path)
    #     point = point.iloc[:, :15]
    #     print(point.shape)
    #     ICESat_points_list.append(point)
    
    #-----------------------------------------------------------------------------------------------
        
    data = pd.concat(ICESat_points_list)
    print(data.shape, 'data shape')
    y_train = data['OH'].values
    x_train = data[target_band].values
    if method == 'RF':
        # classifier = RandomForestRegressor(n_estimators=300, min_samples_leaf=30, random_state=30, max_depth=10)
        # classifier.fit(x_train, y_train)
        best_params, classifier = optimize_random_forest(x_train, y_train)
        classifier.fit(x_train, y_train)
        

C:\Users\minyu\AppData\Local\Temp\ipykernel_13636\2594343571.py:62: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  points.to_file(output_raw_point)
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Unnamed: 0' to 'Unnamed_ 0'
  ogr_write(
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'mean_height' to 'mean_heigh'
  ogr_write(
C:\Users\minyu\AppData\Local\Temp\ipykernel_13636\2594343571.py:63: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  data.to_file(output_point)
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Unnamed: 0' to 'Unnamed_ 0'
  ogr_write(
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normal

(21716, 16) data shape
Best hyperparameters: OrderedDict([('max_depth', 9), ('max_features', 0.1), ('min_samples_leaf', 20), ('min_samples_split', 20), ('n_estimators', 500)])
Best score (negative MSE): -0.4539458940785977


In [ ]:
#----------------------------all ICESat-2 used to train RF modal -----------------------------------------
# Base info
region = 'Region4'

#location_list = ['Region5a', 'Region5b','Region5c', 'Region5d','Region5e']
#location_list = ['Region6a', 'Region6b','Region6c', 'Region6d','Region6e']
#location_list = ['Thames', 'Region1a','Region1b', 'Region1c']
location_list = ['Ribble']
year_list = [2019, 2020, 2021, 2022, 2023]
#year_list = [2020, 2021 ,2022, 2023, 2024]
# region = 'Region2'
# location_list = ['wash_bay', 'Henber', 'Region2a']
# year_list = [2019, 2020, 2021, 2022, 2023, 2024]
# location_list = ['Thames']
# year_list = [2019]
method = 'RF'
band = 4

for i in range(len(location_list)):
    
    location = location_list[i]
    
    LULC_path = "E:/Downloads/tes/UK_LULC.tif"
    LULC = rxr.open_rasterio(LULC_path, masked=True).squeeze()
    target_band_list, target_id_list = ['b1_FB6','b2_FB6','b3_FB6','b4_FB6'], [0, 1, 2, 3]
    target_band, target_id = target_band_list[:band], target_id_list[:band]
    # target_band, target_id = ['b1_FB6'], [0]
    
    
    # load classifer  
    ICESat_points_list = []
    for k in range(len(year_list)):
        
        
        FB_raster_path_ = f"G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}/{location}_{year_list[k]}_FB6_new2.tif"
        ICESAT_csv_path_ = f'G:\ICESat-2\{region}\{region}_{location}\A_{location}_30m_{year_list[k]}_new10.csv'
        output_raw_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_raw_new10.shp'
        output_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_new10.shp'
        data = load_refer_icesat_point(FB_raster_path_, ICESAT_csv_path_, output_point, output_raw_point)
            
        ICESat_points_list.append(data)
        
    #----------------------------------------------------------------------------------------------
    
    # for k in range(len(year_list)):
        
    #     point_path = f'G:\ICESat-2\Discussion5_1\ICESAT\Ribble\A_{location}_c_30m_{year_list[k]}_new10.shp'
    #     point = gpd.read_file(point_path)
    #     point = point.iloc[:, :15]
    #     print(point.shape)
    #     ICESat_points_list.append(point)
    
    #-----------------------------------------------------------------------------------------------
        
    data = pd.concat(ICESat_points_list)
    print(data.shape, 'data shape')
    y_train = data['OH'].values
    x_train = data[target_band].values
    if method == 'RF':
        # classifier = RandomForestRegressor(n_estimators=300, min_samples_leaf=30, random_state=30, max_depth=10)
        # classifier.fit(x_train, y_train)
        best_params, classifier = optimize_random_forest(x_train, y_train)
        classifier.fit(x_train, y_train)
        
    else:
        classifier = np.polyfit(x_train[:, 0], y_train, deg=1) 
        
    #for j in range(len(year_list)):
    for j in range(1):
        # Input info 
        #FB_raster_path = f"G:\ICESat-2\{region}_map\{location}_map\A2022/{location}_{year_list[j]}_FB6_new2.tif"
        FB_raster_path = f'G:\ICESat-2\Discussion5_1\Retrival_DTM\wash_bay_2022_FB6_new2.tif'
        # Output Info
        #output_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[j]}\{location}_{year_list[j]}_{band}_{method}_FB6_new10_A.tif'
        output_path = f'G:\ICESat-2\Discussion5_1\Method_discuss\wash_bay\wash_bay_{year}_{band}_{method}_FB6_new10_bys.tif'
        #output_path =  f'G:\ICESat-2\Discussion5_1\Method_discuss\{region}\All_{region}_AA2019-4_{band}_{method}_FB6_new10.tif'    

        if not os.path.exists(FB_raster_path):
            print(FB_raster_path, ' is not exist!')
            break
       

        # Elevation mapping
        raster = rxr.open_rasterio(FB_raster_path).values
        raster = np.nan_to_num(raster, nan = 0)
        selected_raster = raster[target_id, :, :]

        if len(selected_raster.shape) != 3:
            row, col = selected_raster.shape
            selected_raster = selected_raster.reshape(1, row, col)
        else: 
            bands, row, col = selected_raster.shape

        print('selected_raster.shape', selected_raster.shape)

        k = 0
        result = np.array([[]])
        for i in range(row):
            
            x_train = np.transpose(selected_raster[:, i, :])
            if  method == 'RF':
                y_pred = classifier.predict(x_train)
            else:
                y_pred = np.polyval(classifier, x_train)
            
            result = np.append(result, y_pred)
            
            if i%(int(row/5)) == 0:
                
                print(k*20 , '%_ is done')
                k = k+1
        
        result = result.reshape(row, col)
        print(result.shape, type(result))
        # tf_mask = raster[6,:,:].flatten()
        # result[tf_mask == 0] = 0
        # result_ = result.reshape(row, col)
        tf_mask = raster[4,:,:]#.flatten()
        Freq_s2 = raster[5,:,:]#.flatten()
        
        z = rxr.open_rasterio(FB_raster_path, masked=True).squeeze() 
        LULC_aligned = LULC.rio.reproject_match(z)
        
        #result[(tf_mask == 0) | ((LULC_aligned != 200) & (LULC_aligned != 80))] = 0
        result[tf_mask == 0] = 0

        save_img(output_path, FB_raster_path, result)
        print(f'{location}_{year_list[j]} is output to {output_path}')


In [ ]:
region ='Ribble'
year = 2022
method = 'RF'
#validation_path = f'G:\ICESat-2\Discussion5_1\Validation_point\{region}_{year}_sample_points.shp'
validation_path = f'G:\ICESat-2\Discussion5_1\Validation_point\Validation_2022_Ribble1.shp'
points = gpd.read_file(validation_path)

for i in range(1,5):
    band = i
    val_name = f'RDTM_{band}_{method}'

    Extracted_path =  f'G:\ICESat-2\Discussion5_1\Method_discuss\Ribble\All_Region4_A2019-4_1_RF_FB6_new10.tif'
    with rasterio.open(Extracted_path) as src:
        raster_crs = src.crs  # 获取 raster 的坐标系
        
        # 3. 如果坐标系不同，则重投影点数据
        if points.crs != raster_crs:
        # print(f"坐标系不一致，正在从 {points.crs} 转换为 {raster_crs}")
            points = points.to_crs(raster_crs)

        # 4. 获取坐标并提取像元值（默认波段 1）
        coords = [(geom.x, geom.y) for geom in points.geometry]
        values = [val[0] for val in src.sample(coords)]
    points[val_name] = values
points.to_file(validation_path)

In [ ]:
import geopandas as gpd
year_list = [2019, 2020, 2021, 2022, 2023, 2024]
region = 'Region4'
location = 'Ribble'
# 读取点和面 shapefile
for k in range(len(year_list)):
    point_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_new10.shp'
    output_point_path = f'G:\ICESat-2\Discussion5_1\ICESAT\Ribble\A_{location}_c_30m_{year_list[k]}_new10.shp'
    points = gpd.read_file(point_path)
    polygons = gpd.read_file("G:\ICESat-2\Discussion5_1\Ribble_c.shp")

    # 确保它们使用相同的坐标系
    if points.crs != polygons.crs:
        polygons = polygons.to_crs(points.crs)

    # 执行空间连接（只保留落在 polygon 内的点）
    clipped_points = gpd.sjoin(points, polygons, how="inner", predicate='within')

    # 保存结果
    clipped_points.to_file(output_point_path)

In [35]:
point_path = f'G:\ICESat-2\Discussion5_1\ICESAT\Ribble\A_{location}_c_30m_2023_new10.shp'
point_path2 = f'G:\ICESat-2\Discussion5_1\ICESAT\Ribble\A_{location}_c_30m_2024_new10.shp'
a, b = gpd.read_file(point_path), gpd.read_file(point_path2)
a, b  = a.iloc[:,:18], b.iloc[:,:15]
print(a.columns, a.shape)
print(b.columns)


Index(['Unnamed_ 0', 'distance', 'mean_heigh', 'mean_lon', 'mean_lat', 'OSGM',
       'OH', 'Strip_No', 'b1_FB6', 'b2_FB6', 'b3_FB6', 'b4_FB6', 'b5_FB6',
       'b6_FB6', 'LULC', 'raster_val', 'index_righ', 'Id'],
      dtype='object') (2373, 18)
Index(['Unnamed_ 0', 'distance', 'mean_heigh', 'mean_lon', 'mean_lat', 'OSGM',
       'OH', 'Strip_No', 'b1_FB6', 'b2_FB6', 'b3_FB6', 'b4_FB6', 'b5_FB6',
       'b6_FB6', 'LULC'],
      dtype='object')


In [ ]:
# Base info
region = 'Region2'
location = 'wash_bay'
location_list ['wash_bay']
# location_list = ['Thames', 'Region1a','Region1b','Region1c']
year_list = [2019, 2020, 2021, 2022, 2023, 2024]
# location_list = ['Thames']
#year_list = [2019]
band = 3

for i in range(len(location_list)):
    
    location = location_list[i]
    
    LULC_path = "E:/Downloads/tes/UK_LULC.tif"
    LULC = rxr.open_rasterio(LULC_path, masked=True).squeeze()
    
    for j in range(len(year_list)):
        
        year = year_list[j]
        # Input info 
        #target_band, target_id = ['b1_FB6','b2_FB6','b3_FB6','b4_FB6'], [0, 1, 2, 3]
        target_band, target_id = ['b1_FB6'], [0]
        
        ICESAT_point_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\A_{location}_30m_{year}_new10.shp'
        ICESAT_csv_path = f'G:\ICESat-2\{region}\{region}_{location}\A_{location}_30m_{year}_new10.csv'
        FB_raster_path = f"G:\ICESat-2\{region}_map\{location}_map\A{year}/{location}_{year}_FB6_new2.tif"
        
        # Output Info
        output_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\{location}_{year}_{band}_RF_FB6_new10.tif'
        output_raw_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\A_{location}_30m_{year}_raw_new10.shp'
        output_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\A_{location}_30m_{year}_new10.shp'


        if not os.path.exists(FB_raster_path):
            print(FB_raster_path, ' is not exist!')
            break
            
        # load classifer 
        # if os.path.exists(ICESAT_point_path):
        #     data = gpd.read_file(ICESAT_point_path)
        # else:
        #     data = load_refer_icesat_point(FB_raster_path, ICESAT_csv_path, output_point, output_raw_point)
        data = load_refer_icesat_point(FB_raster_path, ICESAT_csv_path, output_point, output_raw_point)
        
        data = gpd.read_file(ICESAT_point_path)
        y_train = data['OH'].values
        x_train = data[target_band].values
        classifier = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=30)
        classifier.fit(x_train, y_train)

        # Elevation mapping
        raster = rxr.open_rasterio(FB_raster_path).values
        raster = np.nan_to_num(raster, nan = 0)
        selected_raster = raster[target_id, :, :]

        if len(selected_raster.shape) != 3:
            row, col = selected_raster.shape
            selected_raster = selected_raster.reshape(1, row, col)
        else: 
            bands, row, col = selected_raster.shape

        print('selected_raster.shape', selected_raster.shape)

        k = 0
        result = np.array([[]])
        for i in range(row):
            
            x_train = np.transpose(selected_raster[:, i, :])
            y_pred = classifier.predict(x_train)
            
            result = np.append(result, y_pred)
            
            if i%(int(row/5)) == 0:
                
                print(k*20 , '%_ is done')
                k = k+1
        
        result = result.reshape(row, col)
        print(result.shape, type(result))
        # tf_mask = raster[6,:,:].flatten()
        # result[tf_mask == 0] = 0
        # result_ = result.reshape(row, col)
        tf_mask = raster[4,:,:]#.flatten()
        Freq_s2 = raster[5,:,:]#.flatten()
        
        z = rxr.open_rasterio(FB_raster_path, masked=True).squeeze() 
        LULC_aligned = LULC.rio.reproject_match(z)
        
        result[(tf_mask == 0) | (Freq_s2 > 0.95) | ((LULC_aligned != 200) & (LULC_aligned != 80))] = 0
        

        save_img(output_path, FB_raster_path, result)
        print(f'{location}_{year} is output to {output_path}')

In [21]:
#----------------------------all ICESat-2 used to train RF modal -----------------------------------------
# Base info
region = 'Region2'
#location_list = ['Solway', 'Morecambe','Ribble', 'Severn', 'Region4a1', 'Region4a2', 'Region4a3', 'Region4a4', 'Region4b', 'Region4c', 'Region4d']
#location_list = ['Region4a1', 'Region4a2', 'Region4a3', 'Region4a4', 'Region4b', 'Region4c', 'Region4d']
location_list = ['wash_bay']
year_list = [2019, 2020, 2021, 2022, 2023, 2024]
band = 4
method = 'RF'
# region = 'Region2'
# location_list = ['wash_bay', 'Henber', 'Region2a']
# year_list = [2019, 2020, 2021, 2022, 2023, 2024]
# location_list = ['Thames']
# year_list = [2019]


for i in range(len(location_list)):
    
    location = location_list[i]
    
    LULC_path = "E:/Downloads/tes/UK_LULC.tif"
    LULC = rxr.open_rasterio(LULC_path, masked=True).squeeze()
    #target_band, target_id = ['b1_FB6','b2_FB6','b3_FB6','b4_FB6'], [0, 1, 2, 3]
    target_band_list, target_id_list = ['b1_FB6','b2_FB6', 'b3_FB6', 'b4_FB6'], [0, 1, 2, 3]
    target_band, target_id = target_band_list[:band], target_id_list[:band]
    #target_band, target_id = ['b1_FB6', 'b3_FB6'], [0, 2]
    
    # load classifer  
    ICESat_points_list = []
    for k in range(len(year_list)):
        
        
        FB_raster_path_ = f"G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}/{location}_{year_list[k]}_FB6_new2.tif"
        ICESAT_csv_path_ = f'G:\ICESat-2\{region}\{region}_{location}\A_{location}_30m_{year_list[k]}_new10.csv'
        output_raw_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_raw_new10.shp'
        output_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_new10.shp'
        data = load_refer_icesat_point(FB_raster_path_, ICESAT_csv_path_, output_point, output_raw_point)
            
        ICESat_points_list.append(data)
            
    data = pd.concat(ICESat_points_list)
    y_train = data['OH'].values
    x_train = data[target_band].values
    if method == 'RF':
        # classifier = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=30)
        # classifier.fit(x_train, y_train)
        best_params, classifier = optimize_random_forest(x_train, y_train, feature_columns = data[target_band])
        classifier.fit(x_train, y_train)
        
    elif method == 'XGB':
        classifier = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, objective='reg:squarederror', random_state=30, n_jobs=-1)
        classifier.fit(x_train, y_train)
    else:
        classifier = make_pipeline(StandardScaler(), SVR(kernel='rbf', C=100, epsilon=0.1))
        classifier.fit(x_train, y_train)

C:\Users\minyu\AppData\Local\Temp\ipykernel_10600\2186212173.py:62: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  points.to_file(output_raw_point)
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Unnamed: 0' to 'Unnamed_ 0'
  ogr_write(
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'mean_height' to 'mean_heigh'
  ogr_write(
C:\Users\minyu\AppData\Local\Temp\ipykernel_10600\2186212173.py:63: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  data.to_file(output_point)
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Unnamed: 0' to 'Unnamed_ 0'
  ogr_write(
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normal

Best hyperparameters: OrderedDict([('max_depth', 7), ('max_features', 0.663677487543994), ('min_samples_leaf', 1), ('min_samples_split', 20), ('n_estimators', 492)])
Best score (negative MSE): -0.12630911042500859


In [19]:
#----------------------------all ICESat-2 used to train RF modal -----------------------------------------
# Base info
region = 'Region2'
#location_list = ['Solway', 'Morecambe','Ribble', 'Severn', 'Region4a1', 'Region4a2', 'Region4a3', 'Region4a4', 'Region4b', 'Region4c', 'Region4d']
#location_list = ['Region4a1', 'Region4a2', 'Region4a3', 'Region4a4', 'Region4b', 'Region4c', 'Region4d']
location_list = ['wash_bay']
year_list = [2019, 2020, 2021, 2022, 2023, 2024]
band = 3
method = 'RF'
# region = 'Region2'
# location_list = ['wash_bay', 'Henber', 'Region2a']
# year_list = [2019, 2020, 2021, 2022, 2023, 2024]
# location_list = ['Thames']
# year_list = [2019]


for i in range(len(location_list)):
    
    location = location_list[i]
    
    LULC_path = "E:/Downloads/tes/UK_LULC.tif"
    LULC = rxr.open_rasterio(LULC_path, masked=True).squeeze()
    #target_band, target_id = ['b1_FB6','b2_FB6','b3_FB6','b4_FB6'], [0, 1, 2, 3]
    target_band_list, target_id_list = ['b1_FB6','b2_FB6', 'b3_FB6', 'b4_FB6'], [0, 1, 2, 3]
    target_band, target_id = target_band_list[:band], target_id_list[:band]
    #target_band, target_id = ['b1_FB6', 'b3_FB6'], [0, 2]
    
    # load classifer  
    ICESat_points_list = []
    for k in range(len(year_list)):
        
        
        FB_raster_path_ = f"G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}/{location}_{year_list[k]}_FB6_new2.tif"
        ICESAT_csv_path_ = f'G:\ICESat-2\{region}\{region}_{location}\A_{location}_30m_{year_list[k]}_new10.csv'
        output_raw_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_raw_new10.shp'
        output_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_new10.shp'
        data = load_refer_icesat_point(FB_raster_path_, ICESAT_csv_path_, output_point, output_raw_point)
            
        ICESat_points_list.append(data)
            
    data = pd.concat(ICESat_points_list)
    y_train = data['OH'].values
    x_train = data[target_band].values
    if method == 'RF':
        # classifier = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=30)
        # classifier.fit(x_train, y_train)
        best_params, classifier = optimize_random_forest(x_train, y_train, feature_columns = data[target_band])
        classifier.fit(x_train, y_train)
        
    elif method == 'XGB':
        classifier = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, objective='reg:squarederror', random_state=30, n_jobs=-1)
        classifier.fit(x_train, y_train)
    else:
        classifier = make_pipeline(StandardScaler(), SVR(kernel='rbf', C=100, epsilon=0.1))
        classifier.fit(x_train, y_train)
    
    for j in range(len(year_list)):
      
        # Input info 
        FB_raster_path = f"G:\ICESat-2\{region}_map\{location}_map\A{year_list[j]}/{location}_{year_list[j]}_FB6_new2.tif"
    
        # Output Info
        #output_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[j]}\{location}_{year_list[j]}_{band}_{method}_FB6_new10_AAA.tif'
        output_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[j]}\{location}_{year_list[j]}_{band}_{method}_FB6_new10_AAA_bys.tif'

        if not os.path.exists(FB_raster_path):
            print(FB_raster_path, ' is not exist!')
            break
       

        # Elevation mapping
        raster = rxr.open_rasterio(FB_raster_path).values
        raster = np.nan_to_num(raster, nan = 0)
        selected_raster = raster[target_id, :, :]

        if len(selected_raster.shape) != 3:
            row, col = selected_raster.shape
            selected_raster = selected_raster.reshape(1, row, col)
        else: 
            bands, row, col = selected_raster.shape

        print('selected_raster.shape', selected_raster.shape)

        k = 0
        result = np.array([[]])
        for i in range(row):
            
            x_train = np.transpose(selected_raster[:, i, :])
            y_pred = classifier.predict(x_train)
            
            result = np.append(result, y_pred)
            
            if i%(int(row/5)) == 0:
                
                print(k*20 , '%_ is done')
                k = k+1
        
        result = result.reshape(row, col)
        print(result.shape, type(result))
        # tf_mask = raster[6,:,:].flatten()
        # result[tf_mask == 0] = 0
        # result_ = result.reshape(row, col)
        tf_mask = raster[4,:,:]#.flatten()
        Freq_s2 = raster[5,:,:]#.flatten()
        
        z = rxr.open_rasterio(FB_raster_path, masked=True).squeeze() 
        LULC_aligned = LULC.rio.reproject_match(z)
        
        result[(tf_mask == 0) | ((LULC_aligned != 200) & (LULC_aligned != 80))] = 0
        

        save_img(output_path, FB_raster_path, result)
        print(f'{location}_{year_list[j]} is output to {output_path}')

C:\Users\minyu\AppData\Local\Temp\ipykernel_10600\2186212173.py:62: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  points.to_file(output_raw_point)
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Unnamed: 0' to 'Unnamed_ 0'
  ogr_write(
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'mean_height' to 'mean_heigh'
  ogr_write(
C:\Users\minyu\AppData\Local\Temp\ipykernel_10600\2186212173.py:63: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  data.to_file(output_point)
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Unnamed: 0' to 'Unnamed_ 0'
  ogr_write(
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normal

Best hyperparameters: OrderedDict([('max_depth', 11), ('max_features', 0.12254325141473085), ('min_samples_leaf', 1), ('min_samples_split', 19), ('n_estimators', 140)])
Best score (negative MSE): -0.13398894539710351
selected_raster.shape (3, 1592, 1788)
0 %_ is done
20 %_ is done
40 %_ is done
60 %_ is done
80 %_ is done
100 %_ is done
(1592, 1788) <class 'numpy.ndarray'>
wash_bay_2019 is output to G:\ICESat-2\Region2_map\wash_bay_map\A2019\wash_bay_2019_3_RF_FB6_new10_AAA_bys.tif
selected_raster.shape (3, 1592, 1788)
0 %_ is done
20 %_ is done
40 %_ is done
60 %_ is done
80 %_ is done
100 %_ is done
(1592, 1788) <class 'numpy.ndarray'>
wash_bay_2020 is output to G:\ICESat-2\Region2_map\wash_bay_map\A2020\wash_bay_2020_3_RF_FB6_new10_AAA_bys.tif
selected_raster.shape (3, 1592, 1788)
0 %_ is done
20 %_ is done
40 %_ is done
60 %_ is done
80 %_ is done
100 %_ is done
(1592, 1788) <class 'numpy.ndarray'>
wash_bay_2021 is output to G:\ICESat-2\Region2_map\wash_bay_map\A2021\wash_bay_2021_

In [22]:
classifier.feature_importances_

array([0.61603795, 0.01072698, 0.30251429, 0.07072078])

In [ ]:
path = 'E:\Downloads/tes/dem_irl_itm-1.tif'
a = rxr.open_rasterio(path)

In [83]:
#----------------------------all ICESat-2 used to train RF modal -----------------------------------------
# Base info
region = 'Region2'
#location_list = ['Solway', 'Morecambe','Ribble', 'Severn', 'Region4a1', 'Region4a2', 'Region4a3', 'Region4a4', 'Region4b', 'Region4c', 'Region4d']
#location_list = ['Region4a1', 'Region4a2', 'Region4a3', 'Region4a4', 'Region4b', 'Region4c', 'Region4d']
location_list = ['wash_bay']
year_list = [2024]
band = 1
# region = 'Region2'
# location_list = ['wash_bay', 'Henber', 'Region2a']
# year_list = [2019, 2020, 2021, 2022, 2023, 2024]
# location_list = ['Thames']
# year_list = [2019]


for i in range(len(location_list)):
    
    location = location_list[i]
    
    LULC_path = "E:/Downloads/tes/UK_LULC.tif"
    LULC = rxr.open_rasterio(LULC_path, masked=True).squeeze()
    #target_band, target_id = ['b1_FB6','b2_FB6','b3_FB6','b4_FB6'], [0, 1, 2, 3]
    target_band_list, target_id_list = ['b1_FB6','b2_FB6', 'b3_FB6', 'b4_FB6'], [0, 1, 2, 3]
    target_band, target_id = target_band_list[:band], target_id_list[:band]
    
    
    # load classifer  
    ICESat_points_list = []
    for k in range(len(year_list)):
        
        
        FB_raster_path_ = f"G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}/{location}_{year_list[k]}_FB6_new2.tif"
        ICESAT_csv_path_ = f'G:\ICESat-2\{region}\{region}_{location}\A_{location}_30m_{year_list[k]}_new10.csv'
        output_raw_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_raw_new10.shp'
        output_point = f'G:\ICESat-2\{region}_map\{location}_map\A{year_list[k]}\A_{location}_30m_{year_list[k]}_new10.shp'
        data = load_refer_icesat_point(FB_raster_path_, ICESAT_csv_path_, output_point, output_raw_point)
            
        ICESat_points_list.append(data)
            
    data = pd.concat(ICESat_points_list)

C:\Users\minyu\AppData\Local\Temp\ipykernel_26636\2201005165.py:62: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  points.to_file(output_raw_point)
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Unnamed: 0' to 'Unnamed_ 0'
  ogr_write(
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'mean_height' to 'mean_heigh'
  ogr_write(
C:\Users\minyu\AppData\Local\Temp\ipykernel_26636\2201005165.py:63: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  data.to_file(output_point)
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Unnamed: 0' to 'Unnamed_ 0'
  ogr_write(
c:\Users\minyu\.conda\envs\ICESAT_env\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normal

In [1]:
import os
import glob
import shutil

# 输入和输出根目录
input_root = r"E:\FB_download9"
output_root = r"G:\ICESat-2"

# 遍历 reprojected 下所有 TIF 文件
tif_files = glob.glob(os.path.join(input_root, "**", "*.tif"), recursive=True)

print(f"📂 共找到 {len(tif_files)} 个 TIF 文件，开始移动...\n")

for tif in tif_files:
    filename = os.path.basename(tif)
    parent_region = os.path.basename(os.path.dirname(tif))  # 上一级目录名，例如 Region4

    try:
        # 拆分文件名： Region4a2_2024_FB6_new8.tif
        parts = filename.split("_")
        location = parts[0]         # Region4a2
        year = parts[1]             # 2024

        # 构建目标目录
        target_dir = os.path.join(
            output_root,
            f"{parent_region}_map",
            f"{location}_map",
            f"A{year}"
        )
        os.makedirs(target_dir, exist_ok=True)

        # 构建目标文件路径
        target_file = os.path.join(target_dir, filename)

        # 如果已存在同名文件，删除后覆盖
        if os.path.exists(target_file):
            os.remove(target_file)
            print(f"⚠️ 已删除旧文件: {target_file}")

        # 移动文件
        shutil.move(tif, target_file)
        print(f"✅ 已移动: {filename} → {target_file}")

    except Exception as e:
        print(f"❌ 跳过 {filename} ，原因: {e}")

print("\n🎉 所有文件已移动完成！")


📂 共找到 66 个 TIF 文件，开始移动...

✅ 已移动: Region5i_2019_FB6_new9.tif → G:\ICESat-2\Region5_map\Region5i_map\A2019\Region5i_2019_FB6_new9.tif
✅ 已移动: Region5i_2020_FB6_new9.tif → G:\ICESat-2\Region5_map\Region5i_map\A2020\Region5i_2020_FB6_new9.tif
✅ 已移动: Region5i_2021_FB6_new9.tif → G:\ICESat-2\Region5_map\Region5i_map\A2021\Region5i_2021_FB6_new9.tif
✅ 已移动: Region5i_2022_FB6_new9.tif → G:\ICESat-2\Region5_map\Region5i_map\A2022\Region5i_2022_FB6_new9.tif
✅ 已移动: Region5i_2023_FB6_new9.tif → G:\ICESat-2\Region5_map\Region5i_map\A2023\Region5i_2023_FB6_new9.tif
✅ 已移动: Region5i_2024_FB6_new9.tif → G:\ICESat-2\Region5_map\Region5i_map\A2024\Region5i_2024_FB6_new9.tif
✅ 已移动: Region5j_2019_FB6_new9.tif → G:\ICESat-2\Region5_map\Region5j_map\A2019\Region5j_2019_FB6_new9.tif
✅ 已移动: Region5j_2020_FB6_new9.tif → G:\ICESat-2\Region5_map\Region5j_map\A2020\Region5j_2020_FB6_new9.tif
✅ 已移动: Region5j_2021_FB6_new9.tif → G:\ICESat-2\Region5_map\Region5j_map\A2021\Region5j_2021_FB6_new9.tif
✅ 已移动: Region5j_202

In [4]:
import rasterio
import numpy as np

# 输入和输出路径
input_tif = r"G:\ICESat-2\Region2_map\wash_bay_map\A2022\wash_bay_2022_FB6_new9.tif"
output_tif = r"G:\ICESat-2\Region2_map\wash_bay_map\A2022\wash_bay_2022_FB6_new9_masked_.tif"

# 打开原始影像
with rasterio.open(input_tif) as src:
    profile = src.profile.copy()
    
    # 读取所有波段
    data = src.read()  # shape = (bands, height, width)
    
    # 第七个波段作为掩膜
    mask = data[6, :, :]
    
    # 把 NaN 转为 0
    mask = np.nan_to_num(mask, nan=0)
    
    # 构建布尔掩膜：非零区域为 True
    mask_bool = mask != 0
    
    # 对前六个波段进行掩膜
    masked_data = data[:6, :, :].copy()
    masked_data[:, ~mask_bool] = src.nodata if src.nodata is not None else 0
    
    # 更新 profile：波段数从7改为6
    profile.update(count=6)
    
    # 写出结果
    with rasterio.open(output_tif, "w", **profile) as dst:
        dst.write(masked_data)

print("✅ 掩膜完成，输出文件为:", output_tif)


✅ 掩膜完成，输出文件为: G:\ICESat-2\Region2_map\wash_bay_map\A2022\wash_bay_2022_FB6_new9_masked_.tif


In [3]:
data[6, :, :]

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], dtype=float32)